# Grounded Research Crew: CrewAI + Claude + Tavily

A two-agent CrewAI pipeline that researches a topic on the live web and writes a short, **cited** executive summary, followed by an automated **verification step** that checks the summary against the sources the agents actually retrieved.

**Pipeline:** `Researcher (web search) → Writer (summary) → Verifier (deterministic checks)`

**Why the verification step exists.** An earlier version of this project passed only search-result *titles and URLs* to the researcher. The agent never saw article text, yet it produced confident statistics ("stockouts down 65%", "engagement up 40–60%") and named specific companies. Most of that came from the model's prior knowledge, not the sources. This version fixes that in three ways:

1. **Grounding:** the search tool returns page content, and every retrieved source is logged.
2. **Citations:** agents must attach a source URL to every claim.
3. **Verification:** after the run, code checks that every cited URL was really retrieved, every number in the summary appears in a retrieved source, and the word limit is respected.

**Requirements:** an Anthropic API key and a Tavily API key.

In [3]:
# Step 1: Install dependencies (Colab). Locally, use: pip install -r requirements.txt
%pip install -q "crewai[anthropic]" requests nest_asyncio

In [2]:
import os
import re
import datetime as dt

import requests
import nest_asyncio
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool

# Notebook event loops need this patch for CrewAI's async kickoff
nest_asyncio.apply()

## Configuration
API keys are read from Colab Secrets if available, otherwise from environment variables. Never hardcode keys in the notebook.

In [4]:
def load_secret(name: str) -> str:
  """Read a secret from Colab Secrets, falling back to an environment variable."""
  try:
    from google.colab import userdata  # only exists on Colab
    value = userdata.get(name)
  except Exception:
    value = os.environ.get(name)
  if not value:
    raise RuntimeError(f"Missing {name}. Add it to Colab Secrets or set it as an environment variable.")
  return value

os.environ["ANTHROPIC_API_KEY"] = load_secret("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = load_secret("TAVILY_API_KEY")

# ---- Run settings ----
TOPIC = "AI in the FMCG (fast-moving consumer goods) industry"
NUM_FINDINGS = 3
MAX_SUMMARY_WORDS = 100
MODEL = "anthropic/claude-sonnet-4-5-20250929"  # swap for any Claude model your key can access
TODAY = dt.date.today()

## Grounded search tool
The tool returns each result's **content** (not just its title), and records every result in `SOURCE_LOG`. The verifier uses that log later as the ground truth for what the agents were actually shown.

In [5]:
# url -> {"title": ..., "content": ...}; every source the agents were shown
SOURCE_LOG: dict[str, dict] = {}

class TavilySearchTool(BaseTool):
  name: str = "web_search"
  description: str = (
    "Search the web for recent information. Returns numbered results with title, URL, "
    "publish date (if known) and an excerpt of the page content. Input: a search query string."
  )

  def _run(self, query: str) -> str:
    response = requests.post(
      "https://api.tavily.com/search",
      headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
      json={
        "query": query,
        "max_results": 5,
        "search_depth": "advanced",
        "topic": "general",
      },
      timeout=30,
    )
    response.raise_for_status()
    results = response.json().get("results", [])
    if not results:
      return "No web results found."

    blocks = []
    for i, r in enumerate(results, start=1):
      url = r.get("url", "")
      content = (r.get("content") or "").strip()[:1500]
      SOURCE_LOG[url] = {"title": r.get("title", ""), "content": content}
      published = r.get("published_date") or "unknown"
      blocks.append(f"[{i}] {r.get('title', '')}\nURL: {url}\nPublished: {published}\nExcerpt: {content}")
    return "\n\n".join(blocks)

search_tool = TavilySearchTool()

llm = LLM(model=MODEL, max_tokens=3000, temperature=0.2)

## Agents
- The **researcher** has the search tool and enough iterations to finish (the first version hit its `max_iter` cap of 3 and was forced to answer early).
- The **writer** has **no** tools. It may only use the researcher's notes, so it can't introduce new, unsourced material.

In [15]:
researcher = Agent(
  role="AI Industry Researcher",
  goal=f"Find the {NUM_FINDINGS} most significant recent developments in {TOPIC}, backed by sources you actually retrieved.",
  backstory=(
      "You are a careful analyst. You only report facts that appear in the search excerpts you were given, "
      "and you always say where each fact came from. If a source doesn't support a claim, you leave the claim out."
  ),
  tools=[search_tool],
  llm=llm,
  max_iter=10,
  allow_delegation=False,
  verbose=True,
)

writer = Agent(
  role="Executive Technical Writer",
  goal="Turn sourced research notes into a short, accurate executive summary.",
  backstory=(
      "You write for busy executives. You never add facts, numbers or company names that are not in the notes "
      "you were given, and you keep every citation attached to the claim it supports."
  ),
  tools=[],
  llm=llm,
  allow_delegation=False,
  verbose=True,
)

## Tasks
The research task spells out the citation format so the verifier can parse it. The writer task has a **guardrail**: if the summary is over the word limit, CrewAI sends it back to the writer to retry before the task completes.

In [16]:
URL_PATTERN = re.compile(r"https?://[^\s)\]>]+")

def summary_word_count(text: str) -> int:
  """Count words in the summary body, ignoring URLs and markdown symbols."""
  text = URL_PATTERN.sub("", text)
  text = re.sub(r"\(Source:\s*\)", "", text)
  return len(re.findall(r"[A-Za-z0-9$%][\w$%.,'’\-/]*", text))

def word_limit_guardrail(output):
  """CrewAI task guardrail: returns (passed, result_or_feedback)."""

  count = summary_word_count(output.raw)
  if count > MAX_SUMMARY_WORDS:
    return (False, f"The summary is {count} words; the limit is {MAX_SUMMARY_WORDS} (URLs excluded). Shorten it and keep the citations.")
  return (True, output)


research_task = Task(
  description=f"""
  Today's date is {TODAY:%B %d, %Y}. Research: {TOPIC}.

  1. Run several web searches. Focus on developments from the last 12 months and include the current year ({TODAY.year}) in queries where useful.
  2. Pick the {NUM_FINDINGS} most significant developments.
  3. For each one, write notes containing ONLY facts that appear in the search excerpts.

  Rules:
  - Every factual sentence must end with (Source: <exact URL from the search results>).
  - Copy statistics exactly as they appear in the excerpt. Do not round, combine or estimate numbers.
  - Do not name a company as an adopter unless an excerpt says so.
  - If you can't find a supporting source for something you believe is true, leave it out.
  """,
  expected_output=(
    f"Notes on {NUM_FINDINGS} developments. Each has a heading and 2-5 sentences, "
    "and every sentence ends with (Source: URL)."
  ),
  agent=researcher,
)

write_task = Task(
  description=f"""
  Write an executive summary from the research notes.

  Requirements:
  - At most {MAX_SUMMARY_WORDS} words, not counting URLs.
  - Exactly {NUM_FINDINGS} bullet points, one per development.
  - Each bullet ends with (Source: <URL>) using a URL from the notes.
  - Use only facts and numbers that appear in the notes. Do not add anything.
  - No title or preamble; output only the bullets.
  """,
  expected_output=f"{NUM_FINDINGS} cited bullet points, at most {MAX_SUMMARY_WORDS} words.",
  agent=writer,
  context=[research_task],
  guardrail=word_limit_guardrail,
)

## Run the crew

In [17]:
SOURCE_LOG.clear()

crew = Crew(
  agents=[researcher, writer],
  tasks=[research_task, write_task],
  verbose=False,
)

result = await crew.kickoff_async()
summary = result.raw

print(summary)
print(f"\n{len(SOURCE_LOG)} sources retrieved during the run.")

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Industry Researcher                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│    Today's date is October 17, 2026. Research: AI in the FMCG (fast-moving consumer goods) industry.            │
│                                                                                                                 │
│    1. Run several web searches. Focus on developments from the last 12 months and include the current year      │
│  (2026) in queries where useful.                                                                                │
│    2. Pick the 3 most significant developments.                                                                 │
│    3. For each one, write notes containing ONLY facts that appear in the search excerpts.                       │
│                                                                                                                 │
│    Rules:                                                                                                       │
│    - Every factual sentence must end with (Source: <exact URL from the search results>).                        │
│    - Copy statistics exactly as they appear in the excerpt. Do not round, combine or estimate numbers.          │
│    - Do not name a company as an adopter unless an excerpt says so.                                             │
│    - If you can't find a supporting source for something you believe is true, leave it out.                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: [1] Artificial intelligence (AI) In Fast-Moving Consumer Goods
URL: https://www.openpr.com/news/4639640/artificial-intelligence-ai-in-fast-moving-consumer-goods
Published: unknown
Excerpt: The AI mark...
Tool web_search executed with result: [1] AI in FMCG: 16 Statistics on Impact and Use Cases (2026)
URL: https://agilitytech.ai/ai-in-fmcg-statistics
Published: unknown
Excerpt: Data hub · Updated 2026

# The State of AI in FMCG 2026: Key ...
Tool web_search executed with result: [1] Explore the Global Consumer Facing AI Products Market— analysis of key trends, regional growth, top players, and a 10-year forecast from 2026 to 2036.
URL: https://www.futuremarketinsights.com/rep...
Tool web_search executed with result: [1] Machine Learning In Supply Chain: Benefits In 2026
URL: https://retalon.com/blog/machine-learning-in-supply-chain
Published: unknown
Excerpt: Overall, machine learning can help retailers optimize ...
Tool web_search executed with re

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Industry Researcher                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Now I have comprehensive search results. Let me analyze the most significant recent developments in AI in the  │
│  FMCG industry and compile my findings with proper source citations.                                            │
│                                                                                                                 │
│  Based on my research, I'll identify the 3 most significant developments:                                       │
│                                                                                                                 │
│  1. **Agentic AI and Commerce Transformation** - The emergence of agentic AI platforms and protocols like       │
│  Google's Universal Commerce Protocol, with major FMCG companies like Unilever, Nestlé, and PepsiCo             │
│  implementing agentic AI solutions                                                                              │
│  2. **Generative AI for Marketing Content Production** - Companies like Mondelez using generative AI to         │
│  dramatically reduce marketing costs                                                                            │
│  3. **AI-Powered Demand Forecasting and Supply Chain Optimization** - Widespread adoption of AI for demand      │
│  forecasting with significant accuracy improvements                                                             │
│                                                                                                                 │
│  Let me compile the detailed notes with exact source citations:                                                 │
│                                                                                                                 │
│  ## 3 Most Significant Recent Developments in AI in the FMCG Industry                                           │
│                                                                                                                 │
│  ### 1. Agentic AI and the Universal Commerce Protocol Reshape FMCG Commerce                                    │
│                                                                                                                 │
│  Google launched the Universal Commerce Protocol (UCP) on January 11, 2026 at the National Retail Federation    │
│  conference, co-developed with Shopify, Etsy, Wayfair, Target, and Walmart (Source:                             │
│  https://aiadvantageagency.com/universal-commerce-protocol). The UCP is an open standard that enables AI        │
│  agents to execute complete commerce transactions including product discovery, checkout, and post-purchase      │
│  support without a user ever leaving a conversational AI surface (Source:                                       │
│  https://aiadvantageagency.com/universal-commerce-protocol). Unilever announced a five-year partnership with    │
│  Google Cloud in February 2026 to accelerate business transformation using Google Cloud's AI, data, platform    │
│  and marketing capabilities, with the partnership covering agentic commerce and agentic workflows (Source:      │
│  https://www.prnewswire.com/news-releases/unilever-and-google-cloud-partner-to-pioneer-next-generation-of-cons  │
│  umer-goods-technologies-302688857.html). Nestlé's 2025 Annual Report states that its agentic virtual sales     │
│  assistant automates up to 40% of routine tasks in pilo

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Technical Writer                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│    Write an executive summary from the research notes.                                                          │
│                                                                                                                 │
│    Requirements:                                                                                                │
│    - At most 100 words, not counting URLs.                                                                      │
│    - Exactly 3 bullet points, one per development.                                                              │
│    - Each bullet ends with (Source: <URL>) using a URL from the notes.                                          │
│    - Use only facts and numbers that appear in the notes. Do not add anything.                                  │
│    - No title or preamble; output only the bullets.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

Guardrail  blocked (attempt 1/4), retrying due to: The summary is 118 words; the limit is 100 (URLs excluded). Shorten it and keep the citations.



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Technical Writer                                                                              │
│                                                                                                                 │
│  Task:                                                                                                          │
│    Write an executive summary from the research notes.                                                          │
│                                                                                                                 │
│    Requirements:                                                                                                │
│    - At most 100 words, not counting URLs.                                                                      │
│    - Exactly 3 bullet points, one per development.                                                              │
│    - Each bullet ends with (Source: <URL>) using a URL from the notes.                                          │
│    - Use only facts and numbers that appear in the notes. Do not add anything.                                  │
│    - No title or preamble; output only the bullets.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Technical Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  • Google launched the Universal Commerce Protocol in January 2026 for AI agent transactions; Unilever          │
│  partnered with Google Cloud in February 2026 for agentic commerce, and Nestlé's virtual assistant automates    │
│  40% of routine tasks in pilots. (Source:                                                                       │
│  https://www.prnewswire.com/news-releases/unilever-and-google-cloud-partner-to-pioneer-next-generation-of-cons  │
│  umer-goods-technologies-302688857.html)                                                                        │
│                                                                                                                 │
│  • Mondelez uses generative AI to cut marketing costs by 30% to 50%, investing over $40 million in a tool with  │
│  Publicis Groupe and Accenture. (Source:                                                                        │
│  https://www.investing.com/news/stock-market-news/oreomaker-mondelez-to-use-new-generative-ai-tool-to-slash-ma  │
│  rketing-costs-4307008)                                                                                         │
│                                                                                                                 │
│  • AI demand forecasting cuts errors by 20-50% and achieves 92% accuracy versus 75% for traditional methods;    │
│  Danone improved forecast accuracy by 90%. (Source:                                                             │
│  https://syrencloud.com/blogs/fmcg-analytics-growth-use-cases)                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

• Google launched the Universal Commerce Protocol in January 2026 for AI agent transactions; Unilever partnered with Google Cloud in February 2026 for agentic commerce, and Nestlé's virtual assistant automates 40% of routine tasks in pilots. (Source: https://www.prnewswire.com/news-releases/unilever-and-google-cloud-partner-to-pioneer-next-generation-of-consumer-goods-technologies-302688857.html)

• Mondelez uses generative AI to cut marketing costs by 30% to 50%, investing over $40 million in a tool with Publicis Groupe and Accenture. (Source: https://www.investing.com/news/stock-market-news/oreomaker-mondelez-to-use-new-generative-ai-tool-to-slash-marketing-costs-4307008)

• AI demand forecasting cuts errors by 20-50% and achieves 92% accuracy versus 75% for traditional methods; Danone improved forecast accuracy by 90%. (Source: https://syrencloud.com/blogs/fmcg-analytics-growth-use-cases)

67 sources retrieved during the run.


## Verification
Plain Python checks that don't rely on the model grading itself:

| Check | What it catches |
|---|---|
| Word limit | The writer ignoring the length constraint |
| Every bullet has a citation | Unsourced claims |
| Every cited URL was retrieved | Invented or remembered URLs |
| Every number appears in a cited source | Hallucinated or altered statistics |

The number check is intentionally strict: if a figure isn't in the excerpt of the source cited on that bullet, it's flagged for a human to review.

In [18]:
NUMBER_PATTERN = re.compile(r"\$?\d[\d,]*(?:\.\d+)?\s?(?:%|percent|billion|million|bn|x)?", re.IGNORECASE)

def normalize_number(s: str) -> str:
  s = s.lower().replace(",", "").replace(" ", "").replace("percent", "%")
  return s.rstrip(".")

def numbers_in(text: str) -> set[str]:
  return {normalize_number(m.group()) for m in NUMBER_PATTERN.finditer(text) if any(ch.isdigit() for ch in m.group())}

def verify_summary(summary: str, sources: dict[str, dict]) -> list[dict]:
  checks = []

  words = summary_word_count(summary)
  checks.append({"check": f"Word count ≤ {MAX_SUMMARY_WORDS}", "passed": words <= MAX_SUMMARY_WORDS, "detail": f"{words} words"})

  bullets = [ln.strip() for ln in summary.splitlines() if re.match(r"^\s*([-*•]|\d+\.)\s+", ln)]
  checks.append({"check": f"{NUM_FINDINGS} bullets", "passed": len(bullets) == NUM_FINDINGS, "detail": f"{len(bullets)} found"})

  for i, bullet in enumerate(bullets, start=1):
    urls = [u.rstrip(".,;") for u in URL_PATTERN.findall(bullet)]
    checks.append({"check": f"Bullet {i}: has citation", "passed": bool(urls), "detail": ", ".join(urls) or "no URL"})

    unknown = [u for u in urls if u not in sources]
    checks.append({"check": f"Bullet {i}: cited URL was retrieved", "passed": bool(urls) and not unknown, "detail": "not retrieved: " + ", ".join(unknown) if unknown else "ok"})

    cited_text = " ".join(sources[u]["content"] for u in urls if u in sources)
    claim_text = URL_PATTERN.sub("", bullet)
    missing = sorted(numbers_in(claim_text) - numbers_in(cited_text))
    checks.append({"check": f"Bullet {i}: numbers found in cited source", "passed": not missing, "detail": "not found: " + ", ".join(missing) if missing else "ok"})
  return checks

checks = verify_summary(summary, SOURCE_LOG)
for c in checks:
  print(f"{'PASS' if c['passed'] else 'FLAG'}  {c['check']:<42} {c['detail']}")

flagged = [c for c in checks if not c["passed"]]
print(f"\n{len(checks) - len(flagged)}/{len(checks)} checks passed." + (" Review flagged items before using this summary." if flagged else ""))

PASS  Word count ≤ 100                           81 words
PASS  3 bullets                                  3 found
PASS  Bullet 1: has citation                     https://www.prnewswire.com/news-releases/unilever-and-google-cloud-partner-to-pioneer-next-generation-of-consumer-goods-technologies-302688857.html
PASS  Bullet 1: cited URL was retrieved          ok
FLAG  Bullet 1: numbers found in cited source    not found: 40%
PASS  Bullet 2: has citation                     https://www.investing.com/news/stock-market-news/oreomaker-mondelez-to-use-new-generative-ai-tool-to-slash-marketing-costs-4307008
PASS  Bullet 2: cited URL was retrieved          ok
PASS  Bullet 2: numbers found in cited source    ok
PASS  Bullet 3: has citation                     https://syrencloud.com/blogs/fmcg-analytics-growth-use-cases
PASS  Bullet 3: cited URL was retrieved          ok
FLAG  Bullet 3: numbers found in cited source    not found: 20, 50%, 75%, 90%

9/11 checks passed. Review flagged items before

## Sources retrieved
Every page the researcher was shown during the run, for manual spot-checking.

In [19]:
for url, src in SOURCE_LOG.items():
  print(f"- {src['title']}\n  {url}")

- Artificial intelligence (AI) In Fast-Moving Consumer Goods
  https://www.openpr.com/news/4639640/artificial-intelligence-ai-in-fast-moving-consumer-goods
- AI in FMCG: 16 Statistics on Impact and Use Cases (2026)
  https://agilitytech.ai/ai-in-fmcg-statistics
- How AI is Transforming FMCG Supply Chain & Manufacturing in 2026 (Complete Guide) - Dhruv Kubadia
  https://dhruvkubadia.com/how-ai-is-transforming-fmcg-supply-chain-manufacturing-in-2026-complete-guide
- AI in Retail: 10 Trends Reshaping Shopping in 2026
  https://insiderone.com/ai-retail-trends
- Fast Moving Consumer Goods - FMCG Market Analysis 2026: Revenue, Volume, Market Size, Share, CAGR, Growth Trends, Competitor Analysis, Historic Data, and Forecast to 2034
  https://www.cognitivemarketresearch.com/fast-moving-consumer-goods---fmcg-market-report
- Artificial Intelligence in FMCG and Retail Market Size, Share & Forecast Report 2025 to 2034 | Dataintelo
  https://dataintelo.com/report/global-artificial-intelligence-in-f

## Limitations and next steps
- **Excerpt-level grounding.** Verification checks against Tavily's page excerpts, not full articles. A true claim from elsewhere in a page can be flagged; that's a deliberate bias toward false alarms over missed hallucinations.
- **Numbers only.** The verifier checks figures and citations, not whether the *wording* of a claim is faithful. An LLM-as-judge or entailment model could cover that, ideally a different model than the one that wrote the summary.
- **Source quality.** Vendor blogs and market-research landing pages rank well in search but are weak evidence. A domain allowlist or source-quality score would help.
- **Topic is configurable.** Change `TOPIC`, `NUM_FINDINGS` and `MAX_SUMMARY_WORDS` in the configuration cell to reuse the pipeline.